# 03 — Pipeline Validation
Runs the full data pipeline and validates the `model_panel.parquet` output.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
from lamiaty.utils.logging import setup_logging
from lamiaty.config import load_settings
from lamiaty.data.pipeline import run_pipeline
from lamiaty.visualization.diagnostics import plot_missing_heatmap, plot_series_panel

setup_logging()
settings = load_settings("../configs", project_root="..")

## Run the pipeline

In [ ]:
panel = run_pipeline(settings)
print("Panel shape:", panel.shape)
panel.head()

## Missing data heatmap — verify quarterly pattern

In [ ]:
fig = plot_missing_heatmap(
    panel,
    title="Model panel — missing data pattern\n(white = NaN; quarterly series show 2/3 NaN rows)"
)
fig.savefig("../docs/missing_heatmap.png", dpi=150, bbox_inches="tight")

## Verify quarterly series convention

In [ ]:
import pandas as pd
va = panel["va_construction"]
print("VA CONSTRUCTION — NaN count:", va.isna().sum(), "out of", len(va), "rows")
print("\nFirst 12 rows:")
print(va.head(12).to_string())
print("\nExpected: NaN in Jan, Feb; value in Mar (Q1 end)")

## Missing value counts

In [ ]:
panel.isnull().sum().to_frame("n_missing").assign(
    pct_missing=lambda x: (x["n_missing"] / len(panel) * 100).round(1)
)

## Stationarity battery

In [ ]:
from lamiaty.features.stationarity import run_stationarity_battery
battery = run_stationarity_battery(panel.dropna(how="all"))
print(battery[["ADF_pvalue", "KPSS_pvalue", "ADF_stationary", "KPSS_stationary", "verdict"]].to_string())

## Series panel plot

In [ ]:
fig = plot_series_panel(panel, title="Model panel — transformed series")
fig.savefig("../docs/model_panel_series.png", dpi=150, bbox_inches="tight")